In [0]:
# 1. నీ అజూర్ స్టోరేజ్ వివరాలు
storage_account_name = "megacartdatalake01"
container_name = "bronze"
file_name = "customers_batch.csv"

# 🔑 నువ్వు పంపిన అజూర్ యాక్సెస్ కీని ఇక్కడ సెట్ చేసా
storage_key = "azurestoragekey"

# 2. అజూర్ ఫైల్ పాత్ (ADLS Gen2 ABFSS ఎండ్ పాయింట్)
file_path = f"abfss://{container_name}@{storage_account_name}.dfs.core.windows.net/{file_name}"

print(f"🔄 రీడింగ్ ఫైల్: {file_path}")

# 3. డైరెక్ట్‌గా రీడ్ ఆప్షన్స్‌ లోనే కీ ని పాస్ చేసి చదవడం (సర్వర్‌లెస్ ఇన్-లైన్ ట్రిక్)
df = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .option(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", storage_key) \
    .load(file_path)

# 4. సక్సెస్ అయిందో లేదో టాప్ 5 రికార్డులను చూడటం
display(df.limit(5))

🔄 రీడింగ్ ఫైల్: abfss://bronze@megacartdatalake01.dfs.core.windows.net/customers_batch.csv


CustomerId,CustomerName,City,Age,JoinedDate
1001,Aryan Maharaj,Rajahmundry,37,2024-09-01
1002,Gagan Sami,Ludhiana,42,2024-07-27
1003,Viraj Tiwari,Kishanganj,61,2024-10-17
1004,Thomas Sen,Chennai,62,2025-12-26
1005,Pahal Oak,Bhatpara,58,2024-10-30


In [0]:
from pyspark.sql.functions import col, count, when

# 1. పొరపాటున డేటాలో ఎక్కడైనా డూప్లికేట్ రికార్డులు ఉంటే వాటిని తీసేయడం (Data Cleaning)
df_clean = df.dropDuplicates(["CustomerId"])

# 2. Schema Casting & Null Validation (డేటా స్ట్రక్చర్ సరిచేయడం)
# ఒకవేళ ఏవైనా నల్ వాల్యూస్ ఉంటే వాటి స్థానంలో డీఫాల్ట్ వాల్యూస్ పెట్టడం
df_transformed = df_clean.withColumn("Age", col("Age").cast("integer")) \
                         .withColumn("CustomerId", col("CustomerId").cast("integer")) \
                         .withColumn("JoinedDate", col("JoinedDate").cast("date")) \
                         .fillna({"City": "Unknown", "CustomerName": "Unknown"})

# 3. క్లీన్ అయిన డేటాలో ఎన్ని రికార్డులు ఉన్నాయో కౌంట్ చూడటం
total_raw = df.count()
total_clean = df_transformed.count()

print(f"📊 ముడి డేటా రికార్డులు (Raw Count): {total_raw}")
print(f"✨ క్లీన్ చేసిన డేటా రికార్డులు (Clean Count): {total_clean}")
print(f"🗑️ తొలగించబడిన డూప్లికేట్స్ సంఖ్య: {total_raw - total_clean}")

📊 ముడి డేటా రికార్డులు (Raw Count): 5000
✨ క్లీన్ చేసిన డేటా రికార్డులు (Clean Count): 5000
🗑️ తొలగించబడిన డూప్లికేట్స్ సంఖ్య: 0


In [0]:
# సిల్వర్ లేయర్ లోకి సేవ్ చేయబోయే పాత్
silver_path = f"abfss://silver@{storage_account_name}.dfs.core.windows.net/dim_customers"

# ఇన్-లైన్ కీ ఆప్షన్ తో డెల్టా ఫార్మాట్ లో రైట్ చేయడం
df_transformed.write.format("delta") \
    .mode("overwrite") \
    .option(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", storage_key) \
    .save(silver_path)

print("🎉 అద్భుతం! క్లీన్ చేసిన డేటా డెల్టా ఫార్మాట్‌లో Silver Layer కి చేరిపోయింది!")

🎉 అద్భుతం! క్లీన్ చేసిన డేటా డెల్టా ఫార్మాట్‌లో Silver Layer కి చేరిపోయింది!


In [0]:
from pyspark.sql.functions import desc

# 1. సిటీ వైజ్ కస్టమర్ల కౌంట్ ని లెక్కించడం (Business Aggregation)
df_gold = df_transformed.groupBy("City") \
                        .count() \
                        .withColumnRenamed("count", "TotalCustomers") \
                        .orderBy(desc("TotalCustomers"))

# 2. గోల్డ్ డేటా టాప్ 10 సిటీలను స్క్రీన్ మీద చూడటం
display(df_gold.limit(10))

City,TotalCustomers
Aurangabad,38
Ghaziabad,31
Chapra,28
Dhanbad,28
Gangtok,27
Barasat,25
Bhagalpur,25
Siliguri,25
Firozabad,25
North Dumdum,24


In [0]:
# గోల్డ్ లేయర్ లోకి సేవ్ చేయబోయే పాత్
gold_path = f"abfss://gold@{storage_account_name}.dfs.core.windows.net/fact_customer_demographics"

# డెల్టా ఫార్మాట్ లో గోల్డ్ లేయర్ లోకి రైట్ చేయడం
df_gold.write.format("delta") \
    .mode("overwrite") \
    .option(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", storage_key) \
    .save(gold_path)

print("🚀 సక్సెస్! బిజినెస్ రెడీ డేటా డెల్టా ఫార్మాట్‌లో Gold Layer కి చేరిపోయింది. ఎండ్-టు-ఎండ్ పైప్‌లైన్ కంప్లీట్!")

🚀 సక్సెస్! బిజినెస్ రెడీ డేటా డెల్టా ఫార్మాట్‌లో Gold Layer కి చేరిపోయింది. ఎండ్-టు-ఎండ్ పైప్‌లైన్ కంప్లీట్!
